# QELM et MIPT

Point de départ : le TFIM désordonné qui sert de réservoir. La première cellule se place à la racine du dépôt.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in notebooks/. Move to the repository root (the folder that
# contains pyproject.toml) so conf/, data/ and outputs/ resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

In [ ]:
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from QELM import TransverseFieldIsingModel, make_rng, set_seed

GlobalHydra.instance().clear()
with initialize_config_dir(version_base="1.3", config_dir=str(Path("conf").resolve())):
    cfg = compose(config_name="config", overrides=["sim_type=TFIM", "TFIM.N=6"])
set_seed(cfg)
# One generator for the whole run: each model built from it gets new couplings,
# and the same seed gives the same sequence.
rng = make_rng(cfg)
model = TransverseFieldIsingModel.from_config(cfg, rng)
print(f"N={model.N} J={model.J} h={model.h} dim={model.dim}")

In [ ]:
import numpy as np

H = model.hamiltonian()
energies = np.linalg.eigvalsh(H)
print(f"H shape={H.shape}, hermitian={np.allclose(H, H.conj().T)}")
print(f"ground-state energy E0={energies[0]:.4f}, gap={energies[1] - energies[0]:.4f}")

## Disorder average

Each model below draws new couplings from `rng`. The cell builds `rng` from `cfg.seed`, so rerunning it gives the same average.

In [ ]:
n_realisations = 20
rng = make_rng(cfg)
ground = [
    np.linalg.eigvalsh(TransverseFieldIsingModel.from_config(cfg, rng).hamiltonian())[0]
    for _ in range(n_realisations)
]
mean, err = np.mean(ground), np.std(ground) / np.sqrt(n_realisations)
print(f"<E0> over {n_realisations} realisations = {mean:.4f} +/- {err:.4f}")